# Parte 4: Experimentación de Modelado y Ensambles

## Contexto y Estrategia

El dataset de entrenamiento tiene **~800 millones de filas** (~20 GB). Cargar todo en RAM de una vez es imposible incluso con 32 GB, porque pandas necesita memoria adicional para operar sobre los datos. Por eso usamos **dos modos complementarios**:

### Modo 1 — Submuestra para Experimentación Rápida
Usamos el `engineered_sample.csv` generado en el notebook 03 (2.2M filas, ~391 MB). Sirve para:
- Comparar todos los algoritmos rápidamente
- Hacer GridSearch / RandomizedSearch de hiperparámetros
- Elegir el modelo ganador según RMSE en validación

### Modo 2 — Out-of-Core para el Modelo Final
Una vez elegido el ganador, lo entrenamos sobre los **800M de filas reales** usando `fetch_data_in_batches()` de Snowflake. Los boostings (XGBoost, LightGBM, CatBoost) soportan esto nativamente mediante sus APIs de entrenamiento incremental (`xgb.train` con `xgb_model`, `lgb.train` con `init_model`, `CatBoost.fit` con `init_model`).

## Pasos
1. Cargar submuestra y definir splits temporales
2. Pipeline de preprocesamiento
3. Baseline: Regresión Lineal
4. Ensambles: Voting Regressor, Bagging, Pasting
5. Boostings: AdaBoost, GradientBoosting, XGBoost, LightGBM, CatBoost
6. Tabla comparativa RMSE/MAE Train vs Val
7. Evaluación final en Test del modelo ganador
8. Out-of-Core: reentrenar el ganador sobre los 800M de filas
9. Exportar modelo final como `.pkl`

> ⚠️ **Anti-leakage**: El Test Set (2025) se toca **una sola vez**, solo al final, con el modelo ya seleccionado.

---
## 0. Setup e Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import time
import joblib
import sys
import os

# Scikit-learn
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import (
    VotingRegressor, BaggingRegressor,
    AdaBoostRegressor, GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

# Boostings avanzados
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool

# Proyecto
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ Imports completados.')
print(f'  XGBoost:  {xgb.__version__}')
print(f'  LightGBM: {lgb.__version__}')
import catboost
print(f'  CatBoost: {catboost.__version__}')

✅ Imports completados.
  XGBoost:  3.2.0
  LightGBM: 4.6.0
  CatBoost: 1.2.10


---
## 1. Carga de la Submuestra y Configuración

Cargamos el `engineered_sample.csv` del notebook 03 (2.2M filas con todas las features ya construidas) y leemos la configuración de columnas del `feature_config.json`.

In [2]:
# Cargar configuración de features del notebook 03
with open('../data/interim/feature_config.json', 'r') as f:
    feat_cfg = json.load(f)

TARGET             = feat_cfg['target']           # 'total_amount'
NUMERIC_FEATURES   = feat_cfg['numeric_features']
LOCATION_FEATURES  = feat_cfg['location_features']
BINARY_FEATURES    = feat_cfg['binary_features']
CAT_LOW            = feat_cfg['cat_low_card']
CAT_HIGH           = feat_cfg['cat_high_card']
ALL_FEATURES       = feat_cfg['all_features']

print(f'Target: {TARGET}')
print(f'Total features: {len(ALL_FEATURES)}')
print(f'  Numéricas:          {len(NUMERIC_FEATURES)}')
print(f'  Ubicación (IDs):    {len(LOCATION_FEATURES)}')
print(f'  Binarias:           {len(BINARY_FEATURES)}')
print(f'  Categ. baja card.:  {len(CAT_LOW)}')
print(f'  Categ. alta card.:  {len(CAT_HIGH)}')

Target: total_amount
Total features: 34
  Numéricas:          9
  Ubicación (IDs):    2
  Binarias:           15
  Categ. baja card.:  6
  Categ. alta card.:  2


In [3]:
# Cargar submuestra con features ya construidas
df = pd.read_csv('../data/interim/engineered_sample.csv')

# Asegurar que pickup_datetime es datetime
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')

print(f'✅ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Memoria: {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
print(f'Rango temporal: {df["year"].min()} – {df["year"].max()}')

✅ Dataset cargado: 2,274,211 filas × 47 columnas
Memoria: 1103.5 MB
Rango temporal: 2015 – 2023


---
## 2. Split Temporal (No Aleatorio)

El split debe ser **temporal**, nunca aleatorio. Usar `train_test_split` aleatorio causaría **Data Leakage** porque el modelo vería datos futuros durante el entrenamiento.

- **Train**: 2015–2022 (dentro de la submuestra)
- **Validación**: 2023 (el año más reciente disponible en la submuestra)
- **Test**: Se consume desde Snowflake (`analytics.test_set`, año 2025) — solo al final

> Nota: La submuestra es del `train_set` de Snowflake (2015–2023). Usamos 2023 como validación local.

In [4]:
# Filtrar solo las features disponibles en el dataframe
available_features = [f for f in ALL_FEATURES if f in df.columns]
missing = [f for f in ALL_FEATURES if f not in df.columns]
if missing:
    print(f'⚠️ Features faltantes en el CSV: {missing}')

# Split temporal dentro de la submuestra
train_mask = df['year'] <= 2022
val_mask   = df['year'] == 2023

X_train = df.loc[train_mask, available_features]
y_train = df.loc[train_mask, TARGET]

X_val   = df.loc[val_mask, available_features]
y_val   = df.loc[val_mask, TARGET]

print('=== SPLIT TEMPORAL ===')
print(f'Train (2015-2022): {len(X_train):,} filas ({len(X_train)/len(df)*100:.1f}%)')
print(f'Val   (2023):      {len(X_val):,} filas ({len(X_val)/len(df)*100:.1f}%)')
print(f'\ny_train — Media: ${y_train.mean():.2f} | Std: ${y_train.std():.2f}')
print(f'y_val   — Media: ${y_val.mean():.2f} | Std: ${y_val.std():.2f}')

=== SPLIT TEMPORAL ===
Train (2015-2022): 2,165,284 filas (95.2%)
Val   (2023):      108,927 filas (4.8%)

y_train — Media: $16.82 | Std: $13.74
y_val   — Media: $28.84 | Std: $22.79


---
## 3. Pipelines de Preprocesamiento

Definimos **dos preprocessors** según el tipo de modelo:

- **`preprocessor_trees`**: Para árboles y boostings. Sin `StandardScaler` (los árboles son invariantes a la escala). Usa `OrdinalEncoder` para categóricas.
- **`preprocessor_linear`**: Para regresión lineal (baseline). Con `StandardScaler` y `OneHotEncoder`.

Ambos manejan nulos residuales con `SimpleImputer`.

In [5]:
from src.features.build_features import (
    build_preprocessor_for_trees,
    build_preprocessor_for_linear,
)

# Filtrar listas a columnas disponibles
num_feats = [c for c in NUMERIC_FEATURES  if c in available_features]
loc_feats = [c for c in LOCATION_FEATURES if c in available_features]
bin_feats = [c for c in BINARY_FEATURES   if c in available_features]
cat_low   = [c for c in CAT_LOW           if c in available_features]
cat_high  = [c for c in CAT_HIGH          if c in available_features]

preprocessor_trees  = build_preprocessor_for_trees(num_feats, loc_feats, bin_feats, cat_low, cat_high)
preprocessor_linear = build_preprocessor_for_linear(num_feats, bin_feats, cat_low)

print('✅ Preprocessors listos.')
print(f'  Trees  → columnas de entrada: {len(available_features)}')
print(f'  Linear → columnas de entrada: {len(num_feats + bin_feats + cat_low)}')

✅ Preprocessors listos.
  Trees  → columnas de entrada: 34
  Linear → columnas de entrada: 30


---
## 4. Utilidades de Evaluación

Centralizamos la evaluación en una función para que todos los modelos se comparen con las mismas métricas:
- **RMSE** (Root Mean Squared Error): métrica principal del profe, penaliza errores grandes
- **MAE** (Mean Absolute Error): más interpretable en dólares
- **R²**: qué porcentaje de la varianza explica el modelo

In [6]:
# Tabla global de resultados
results = []

def evaluate_model(name, model, X_tr, y_tr, X_v, y_v, fit_time=None):
    """
    Evalúa un modelo en train y validación, registra las métricas
    y las agrega a la tabla global de resultados.
    
    Args:
        name: Nombre del modelo para la tabla comparativa.
        model: Modelo ya entrenado (con .predict()).
        X_tr, y_tr: Datos de entrenamiento.
        X_v, y_v: Datos de validación.
        fit_time: Tiempo de entrenamiento en segundos (opcional).
    
    Returns:
        dict con las métricas calculadas.
    """
    pred_train = model.predict(X_tr)
    pred_val   = model.predict(X_v)

    metrics = {
        'Modelo':        name,
        'RMSE Train':    np.sqrt(mean_squared_error(y_tr, pred_train)),
        'RMSE Val':      np.sqrt(mean_squared_error(y_v,  pred_val)),
        'MAE Val':       mean_absolute_error(y_v, pred_val),
        'R² Val':        r2_score(y_v, pred_val),
        'Tiempo (s)':    round(fit_time, 1) if fit_time else None,
    }
    results.append(metrics)

    print(f'  RMSE Train: ${metrics["RMSE Train"]:>7.3f}')
    print(f'  RMSE Val:   ${metrics["RMSE Val"]:>7.3f}  ← métrica principal')
    print(f'  MAE Val:    ${metrics["MAE Val"]:>7.3f}')
    print(f'  R² Val:      {metrics["R² Val"]:>7.4f}')
    if fit_time:
        print(f'  Tiempo:      {fit_time:.1f}s')
    return metrics

print('✅ Función de evaluación lista.')

✅ Función de evaluación lista.


---
## 5. Baseline: Regresión Lineal

El baseline establece el piso mínimo de rendimiento. **Cualquier modelo más complejo debe superar este RMSE** para justificar su complejidad adicional.

Usamos `preprocessor_linear` (con `StandardScaler` + `OneHotEncoder`) porque la regresión lineal sí requiere escala y variables dummy.

In [7]:
print('=== BASELINE: Regresión Lineal ===')

# Las features del modelo lineal son un subconjunto (sin alta cardinalidad)
lin_feats = num_feats + bin_feats + cat_low

pipeline_linear = Pipeline([
    ('preproc', preprocessor_linear),
    ('model',   LinearRegression(n_jobs=-1)),
])

t0 = time.time()
pipeline_linear.fit(X_train[lin_feats], y_train)
fit_time = time.time() - t0

evaluate_model(
    'Regresión Lineal (Baseline)',
    pipeline_linear,
    X_train[lin_feats], y_train,
    X_val[lin_feats],   y_val,
    fit_time
)

=== BASELINE: Regresión Lineal ===
  RMSE Train: $  4.601
  RMSE Val:   $ 13.223  ← métrica principal
  MAE Val:    $  9.613
  R² Val:       0.6635
  Tiempo:      20.7s


{'Modelo': 'Regresión Lineal (Baseline)',
 'RMSE Train': np.float64(4.601445283672045),
 'RMSE Val': np.float64(13.22304670301012),
 'MAE Val': 9.612770693984746,
 'R² Val': 0.663491361110979,
 'Tiempo (s)': 20.7}

---
## 6. Ensambles: Voting Regressor

El **Voting Regressor** combina las predicciones de múltiples modelos base promediando sus salidas. La idea es que el error aleatorio de cada modelo se cancela parcialmente al promediar, reduciendo la varianza total.

Usamos estimadores de diferente naturaleza (árboles + lineal) para maximizar la diversidad, que es el ingrediente clave del voting.

In [8]:
print('=== ENSAMBLE: Voting Regressor ===')
print('Combinando Ridge, DecisionTree y RandomForest por promedio simple.')
print('La diversidad entre estimadores es clave para que el voting reduzca el error.')

from sklearn.linear_model import Ridge

# Preprocesar una vez para los modelos de árbol
X_train_t = preprocessor_trees.fit_transform(X_train[available_features])
X_val_t   = preprocessor_trees.transform(X_val[available_features])

# Estimadores base del voting (ya trabajan sobre datos transformados)
estimators = [
    ('ridge', Ridge(alpha=1.0)),
    ('tree',  DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE)),
    ('rf',    RandomForestRegressor(n_estimators=50, max_depth=8,
                                     n_jobs=-1, random_state=RANDOM_STATE)),
]

voting = VotingRegressor(estimators=estimators, n_jobs=-1)

t0 = time.time()
voting.fit(X_train_t, y_train)
fit_time = time.time() - t0

evaluate_model('Voting Regressor', voting,
               X_train_t, y_train, X_val_t, y_val, fit_time)

=== ENSAMBLE: Voting Regressor ===
Combinando Ridge, DecisionTree y RandomForest por promedio simple.
La diversidad entre estimadores es clave para que el voting reduzca el error.
  RMSE Train: $  3.933
  RMSE Val:   $ 12.945  ← métrica principal
  MAE Val:    $  9.753
  R² Val:       0.6775
  Tiempo:      218.1s


{'Modelo': 'Voting Regressor',
 'RMSE Train': np.float64(3.9331618855899375),
 'RMSE Val': np.float64(12.944562333817021),
 'MAE Val': 9.752605737260396,
 'R² Val': 0.6775162045161112,
 'Tiempo (s)': 218.1}

---
## 7. Ensambles: Bagging vs Pasting

**Bagging** y **Pasting** entrenan múltiples instancias del mismo estimador base sobre subsets distintos del dataset.

- **Bagging** (`bootstrap=True`): cada estimador entrena sobre una muestra **con reemplazo**. Reduce la varianza del estimador base.
- **Pasting** (`bootstrap=False`): muestra **sin reemplazo**. Cada muestra es única, más diversidad pero sin el efecto regularizador del bootstrap.

Comparamos ambos con el mismo árbol base para aislar el efecto del bootstrap.

In [9]:
print('=== ENSAMBLE: Bagging ===')
print('bootstrap=True → cada árbol entrena sobre muestra CON reemplazo.')
print('Efecto: reduce varianza del estimador base (árboles profundos tienden a overfitear).')

bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
    n_estimators=50,
    max_samples=0.8,    # cada árbol usa el 80% de las filas
    max_features=0.8,   # cada árbol usa el 80% de las features
    bootstrap=True,     # con reemplazo = BAGGING
    n_jobs=-1,
    random_state=RANDOM_STATE
)

t0 = time.time()
bagging.fit(X_train_t, y_train)
fit_time = time.time() - t0

evaluate_model('Bagging (bootstrap=True)', bagging,
               X_train_t, y_train, X_val_t, y_val, fit_time)

=== ENSAMBLE: Bagging ===
bootstrap=True → cada árbol entrena sobre muestra CON reemplazo.
Efecto: reduce varianza del estimador base (árboles profundos tienden a overfitear).
  RMSE Train: $  3.579
  RMSE Val:   $ 12.879  ← métrica principal
  MAE Val:    $  9.833
  R² Val:       0.6808
  Tiempo:      235.4s


{'Modelo': 'Bagging (bootstrap=True)',
 'RMSE Train': np.float64(3.5793931634650535),
 'RMSE Val': np.float64(12.878936197245222),
 'MAE Val': 9.833392742308623,
 'R² Val': 0.6807777622594775,
 'Tiempo (s)': 235.4}

In [10]:
print('=== ENSAMBLE: Pasting ===')
print('bootstrap=False → cada árbol entrena sobre muestra SIN reemplazo.')
print('Cada subconjunto es único. Más diversidad pero sin el efecto regularizador del bootstrap.')

pasting = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=False,    # sin reemplazo = PASTING
    n_jobs=-1,
    random_state=RANDOM_STATE
)

t0 = time.time()
pasting.fit(X_train_t, y_train)
fit_time = time.time() - t0

evaluate_model('Pasting (bootstrap=False)', pasting,
               X_train_t, y_train, X_val_t, y_val, fit_time)

=== ENSAMBLE: Pasting ===
bootstrap=False → cada árbol entrena sobre muestra SIN reemplazo.
Cada subconjunto es único. Más diversidad pero sin el efecto regularizador del bootstrap.
  RMSE Train: $  3.562
  RMSE Val:   $ 12.892  ← métrica principal
  MAE Val:    $  9.838
  R² Val:       0.6801
  Tiempo:      1494.6s


{'Modelo': 'Pasting (bootstrap=False)',
 'RMSE Train': np.float64(3.5616173621412623),
 'RMSE Val': np.float64(12.891642909346999),
 'MAE Val': 9.837518051261249,
 'R² Val': 0.6801475446668827,
 'Tiempo (s)': 1494.6}

---
## 8. Boosting 1: AdaBoost

**AdaBoost** (Adaptive Boosting) es el boosting original. Entrena estimadores secuencialmente: cada uno se enfoca en los ejemplos que el anterior predijo **peor**, aumentando su peso en la siguiente iteración.

**Parámetros clave:**
- `n_estimators`: número de estimadores secuenciales. Más → menor bias, pero riesgo de overfitting
- `learning_rate`: cuánto contribuye cada estimador. Trade-off con `n_estimators` (más estimadores → menor learning_rate)
- `max_depth` del árbol base: AdaBoost clásico usa `stumps` (depth=1), pero depth=3-4 mejora con datos complejos

**Limitación vs otros boostings:** AdaBoost es sensible a outliers porque los re-pondera agresivamente.

In [ ]:
print('=== BOOSTING 1: AdaBoost ===')
print('Entrenamiento secuencial: cada árbol corrige los errores del anterior.')
print('Usa stumps (árboles de profundidad 3) como estimadores base.')

ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=3),
    n_estimators=100,
    learning_rate=0.1,   # contribución de cada estimador; menos → más regularización
    loss='linear',       # función de pérdida para actualizar pesos: linear, square, exponential
    random_state=RANDOM_STATE
)

t0 = time.time()
ada.fit(X_train_t, y_train)
fit_time = time.time() - t0

evaluate_model('AdaBoost', ada,
               X_train_t, y_train, X_val_t, y_val, fit_time)

=== BOOSTING 1: AdaBoost ===
Entrenamiento secuencial: cada árbol corrige los errores del anterior.
Usa stumps (árboles de profundidad 3) como estimadores base.


---
## 9. Boosting 2: Gradient Boosting (GBDT)

**Gradient Boosting** generaliza el concepto de boosting: en lugar de re-ponderar ejemplos, cada árbol nuevo aproxima el **gradiente negativo** de la función de pérdida (los residuos del modelo anterior).

**Parámetros clave:**
- `n_estimators`: número de árboles. Con `learning_rate` bajo se necesitan más árboles
- `learning_rate`: shrinkage — cuánto se "encoge" la contribución de cada árbol. Menor → más generalización
- `max_depth`: profundidad de cada árbol. GBDT clásico usa árboles poco profundos (3-5)
- `subsample`: fracción de filas por árbol (< 1.0 introduce Stochastic GBM, reduce varianza)
- `min_samples_leaf`: regularización para evitar splits con pocas muestras

In [ ]:
print('=== BOOSTING 2: Gradient Boosting (GBDT) ===')
print('Cada árbol ajusta los residuos (pseudo-residuos = gradiente de la pérdida) del anterior.')
print('subsample < 1.0 → Stochastic GBM, reduce varianza al introducir aleatoriedad.')

gbdt = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,   # bajo learning_rate → más árboles necesarios pero mejor generalización
    max_depth=4,
    subsample=0.8,         # 80% de filas por árbol (Stochastic GBM)
    min_samples_leaf=50,   # mínimo de muestras en cada hoja (regularización)
    max_features='sqrt',   # features por split (reduce varianza)
    random_state=RANDOM_STATE
)

t0 = time.time()
gbdt.fit(X_train_t, y_train)
fit_time = time.time() - t0

evaluate_model('Gradient Boosting (GBDT)', gbdt,
               X_train_t, y_train, X_val_t, y_val, fit_time)

---
## 10. Boosting 3: XGBoost

**XGBoost** (Extreme Gradient Boosting) es una implementación optimizada de GBDT con varias mejoras:
- **Regularización L1/L2** (`alpha`, `lambda`) integrada en la función objetivo
- **Parallel split finding**: evalúa splits en paralelo (mucho más rápido que sklearn GBDT)
- **Sparse-aware**: maneja nulos nativamente sin imputación
- **Tree pruning**: poda ramas que no mejoran el objetivo

Para **Out-of-Core**, XGBoost soporta entrenamiento incremental pasando el modelo anterior como `xgb_model` en cada nuevo batch.

In [ ]:
print('=== BOOSTING 3: XGBoost ===')
print('Regularización L1/L2 integrada, sparse-aware, evaluación de splits en paralelo.')
print('tree_method="hist" es el más eficiente para datasets grandes (agrupa valores en bins).')

xgboost_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,  # fracción de features por árbol
    reg_alpha=0.1,         # regularización L1 (lasso): lleva pesos a cero
    reg_lambda=1.0,        # regularización L2 (ridge): penaliza pesos grandes
    tree_method='hist',    # histograma de splits: más rápido para datasets grandes
    n_jobs=-1,
    random_state=RANDOM_STATE,
    early_stopping_rounds=20,  # detiene si val no mejora en 20 rondas
    eval_metric='rmse',
    verbosity=0
)

t0 = time.time()
xgboost_model.fit(
    X_train_t, y_train,
    eval_set=[(X_val_t, y_val)],
    verbose=False
)
fit_time = time.time() - t0

print(f'  Mejor ronda: {xgboost_model.best_iteration} (early stopping activo)')
evaluate_model('XGBoost', xgboost_model,
               X_train_t, y_train, X_val_t, y_val, fit_time)

---
## 11. Boosting 4: LightGBM

**LightGBM** es el boosting más rápido para datasets grandes. Sus dos innovaciones principales:

1. **GOSS** (Gradient-based One-Side Sampling): descarta instancias con gradiente pequeño (ya bien predichas), manteniendo solo las difíciles. Reduce datos sin perder información importante.
2. **EFB** (Exclusive Feature Bundling): agrupa features mutuamente excluyentes (muchos ceros) en una sola. Reduce dimensionalidad efectiva.
3. **Crecimiento por hoja** (`num_leaves`): a diferencia de GBDT que crece por nivel, LightGBM crece por la hoja de mayor ganancia → árboles más asimétricos y eficientes.

Para **Out-of-Core**, LightGBM acepta `init_model` para continuar entrenamiento desde un modelo guardado.

In [ ]:
print('=== BOOSTING 4: LightGBM ===')
print('GOSS sampling + EFB bundling → el más rápido para datasets grandes.')
print('Crece por hoja (leaf-wise) en vez de por nivel (level-wise) como GBDT.')

lgbm_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,          # máximo de hojas por árbol (más hojas → más complejo)
    max_depth=-1,           # sin límite de profundidad (controlado por num_leaves)
    min_child_samples=50,   # mínimo de muestras por hoja (regularización)
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=-1
)

callbacks = [lgb.early_stopping(stopping_rounds=20, verbose=False),
             lgb.log_evaluation(period=-1)]

t0 = time.time()
lgbm_model.fit(
    X_train_t, y_train,
    eval_set=[(X_val_t, y_val)],
    callbacks=callbacks
)
fit_time = time.time() - t0

print(f'  Mejor iteración: {lgbm_model.best_iteration_}')
evaluate_model('LightGBM', lgbm_model,
               X_train_t, y_train, X_val_t, y_val, fit_time)

---
## 12. Boosting 5: CatBoost

**CatBoost** (Categorical Boosting) es el boosting de Yandex. Sus ventajas principales:

1. **Manejo nativo de categóricas**: no necesita OrdinalEncoder ni OneHotEncoder — internamente usa **Target Statistics** (encoding basado en el target) que es más informativo y evita leakage dentro del entrenamiento usando ordered boosting.
2. **Ordered Boosting**: para evitar *prediction shift* (sesgo por usar el mismo dato para calcular residuos y para la predicción), CatBoost usa permutaciones ordenadas.
3. **Symmetric trees**: árboles balanceados por construcción → más rápidos para inferencia.

Al pasarle `cat_features`, CatBoost maneja las categóricas sin necesidad del OrdinalEncoder.

In [ ]:
print('=== BOOSTING 5: CatBoost ===')
print('Manejo nativo de categóricas (Target Statistics + Ordered Boosting).')
print('Symmetric trees → inferencia más rápida en producción.')
print('No necesita OrdinalEncoder: recibe las categóricas como strings directamente.')

# CatBoost trabaja directamente con el DataFrame sin preprocesar las categóricas
# Solo necesita saber cuáles columnas son categóricas
cat_feature_names = cat_low + cat_high
cat_feature_names = [c for c in cat_feature_names if c in available_features]

# Preparar datos: CatBoost necesita strings (no category dtype)
X_train_cat = X_train[available_features].copy()
X_val_cat   = X_val[available_features].copy()
for col in cat_feature_names:
    X_train_cat[col] = X_train_cat[col].astype(str).fillna('Unknown')
    X_val_cat[col]   = X_val_cat[col].astype(str).fillna('Unknown')

# Índices de columnas categóricas (CatBoost las necesita por posición o nombre)
cat_indices = [list(available_features).index(c) for c in cat_feature_names
               if c in available_features]

catboost_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,        # regularización L2 en las hojas
    bagging_temperature=1,  # aleatoriedad tipo bayesiano en el bootstrap
    border_count=128,       # bins para features numéricas (como hist en XGB)
    early_stopping_rounds=20,
    eval_metric='RMSE',
    random_seed=RANDOM_STATE,
    verbose=0
)

t0 = time.time()
catboost_model.fit(
    X_train_cat, y_train,
    cat_features=cat_feature_names,
    eval_set=(X_val_cat, y_val),
    verbose=False
)
fit_time = time.time() - t0

print(f'  Mejor iteración: {catboost_model.best_iteration_}')
evaluate_model('CatBoost', catboost_model,
               X_train_cat, y_train, X_val_cat, y_val, fit_time)

---
## 13. Tabla Comparativa y Selección del Ganador

In [ ]:
results_df = pd.DataFrame(results).sort_values('RMSE Val')

print('=' * 75)
print('TABLA COMPARATIVA DE MODELOS (ordenado por RMSE Val)')
print('=' * 75)
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('=' * 75)

ganador = results_df.iloc[0]
baseline_rmse = results_df[results_df['Modelo'] == 'Regresión Lineal (Baseline)']['RMSE Val'].values[0]
mejora = (baseline_rmse - ganador['RMSE Val']) / baseline_rmse * 100

print(f'\n🏆 GANADOR: {ganador["Modelo"]}')
print(f'   RMSE Val:  ${ganador["RMSE Val"]:.4f}')
print(f'   Mejora vs Baseline: {mejora:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: RMSE Val comparativo
colors = ['#e74c3c' if m == ganador['Modelo'] else '#3498db' for m in results_df['Modelo']]
bars = axes[0].barh(results_df['Modelo'], results_df['RMSE Val'],
                    color=colors, alpha=0.85, edgecolor='white')
axes[0].set_title('RMSE en Validación por Modelo\n(menor es mejor)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('RMSE ($)')
for bar, val in zip(bars, results_df['RMSE Val']):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'${val:.3f}', va='center', fontsize=9)

# Gráfica 2: RMSE Train vs Val (diagnóstico de overfitting)
x = range(len(results_df))
axes[1].plot(x, results_df['RMSE Train'], 'o-', color='steelblue', label='RMSE Train', linewidth=2)
axes[1].plot(x, results_df['RMSE Val'],   's--', color='coral',    label='RMSE Val',   linewidth=2)
axes[1].set_xticks(x)
axes[1].set_xticklabels(results_df['Modelo'], rotation=30, ha='right', fontsize=9)
axes[1].set_title('Train vs Val RMSE\n(gap grande = overfitting)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('RMSE ($)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/interim/04_model_comparison.png', bbox_inches='tight', dpi=120)
plt.show()

---
## 14. Evaluación Final en Test Set

> ⚠️ **Esta celda se corre UNA sola vez, con el modelo ya elegido.** Ejecutar esto antes de seleccionar el ganador constituiría Data Leakage (usar el test para elegir el modelo).

El Test Set viene directamente de Snowflake (`analytics.test_set`, año 2025). Aplicamos el mismo pipeline de feature engineering antes de predecir.

In [ ]:
from src.data.ingestion import fetch_sample
from src.features.build_features import clean_dataframe, run_feature_engineering

print('Cargando Test Set (2025) desde Snowflake...')
query_test = "SELECT * FROM NYC_TAXI_P5.ANALYTICS.test_set"

# Muestra del 1% del test (suficiente para evaluación final representativa)
df_test_raw = fetch_sample(query_test, sample_prob=1.0)
print(f'Test crudo: {df_test_raw.shape[0]:,} filas')

# Aplicar el mismo pipeline de limpieza y feature engineering
df_test_clean = clean_dataframe(df_test_raw)
df_test_fe    = run_feature_engineering(df_test_clean)

test_feats = [f for f in available_features if f in df_test_fe.columns]

X_test = df_test_fe[test_feats]
y_test = df_test_fe[TARGET]

print(f'Test procesado: {X_test.shape[0]:,} filas × {X_test.shape[1]} features')

In [ ]:
# ─── Cambiar aquí según el modelo ganador ───
# Opciones: xgboost_model, lgbm_model, catboost_model, gbdt, ada, voting, bagging, pasting
WINNER_NAME  = ganador['Modelo']  # se asigna automáticamente desde la tabla

# Mapear nombre a objeto modelo
model_map = {
    'XGBoost':                  xgboost_model,
    'LightGBM':                 lgbm_model,
    'CatBoost':                 catboost_model,
    'Gradient Boosting (GBDT)': gbdt,
    'AdaBoost':                 ada,
    'Voting Regressor':         voting,
    'Bagging (bootstrap=True)': bagging,
    'Pasting (bootstrap=False)': pasting,
}

winner_model = model_map.get(WINNER_NAME)

# Preprocesar test según el tipo de modelo ganador
if WINNER_NAME == 'CatBoost':
    X_test_proc = X_test.copy()
    for col in cat_feature_names:
        if col in X_test_proc.columns:
            X_test_proc[col] = X_test_proc[col].astype(str).fillna('Unknown')
else:
    X_test_proc = preprocessor_trees.transform(X_test)

pred_test = winner_model.predict(X_test_proc)

rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
mae_test  = mean_absolute_error(y_test, pred_test)
r2_test   = r2_score(y_test, pred_test)

print('=' * 50)
print(f'EVALUACIÓN FINAL EN TEST SET (2025)')
print(f'Modelo: {WINNER_NAME}')
print('=' * 50)
print(f'  RMSE Test: ${rmse_test:.4f}')
print(f'  MAE  Test: ${mae_test:.4f}')
print(f'  R²   Test:  {r2_test:.4f}')
print('=' * 50)

In [ ]:
# Gráfica de residuos: diagnóstico de calidad del modelo ganador
residuos = y_test.values - pred_test

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribución de residuos
axes[0].hist(residuos, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Distribución de Residuos', fontsize=13)
axes[0].set_xlabel('Residuo ($)')
axes[0].set_ylabel('Frecuencia')

# Predicho vs Real
sample_idx = np.random.choice(len(y_test), min(10000, len(y_test)), replace=False)
axes[1].scatter(y_test.values[sample_idx], pred_test[sample_idx],
                alpha=0.2, s=2, color='steelblue')
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[1].set_title('Predicho vs Real', fontsize=13)
axes[1].set_xlabel('Real ($)')
axes[1].set_ylabel('Predicho ($)')
axes[1].legend()

# Residuos vs Predicho
axes[2].scatter(pred_test[sample_idx], residuos[sample_idx],
                alpha=0.2, s=2, color='coral')
axes[2].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[2].set_title('Residuos vs Predicho', fontsize=13)
axes[2].set_xlabel('Predicho ($)')
axes[2].set_ylabel('Residuo ($)')

plt.suptitle(f'Diagnóstico de Residuos — {WINNER_NAME}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/interim/04_residuals.png', bbox_inches='tight', dpi=120)
plt.show()

---
## 15. Out-of-Core: Reentrenar el Ganador sobre los 800M de Filas

### ¿Qué es Out-of-Core Training?

Out-of-Core (OOC) significa entrenar un modelo sobre datos que **no caben en RAM**, procesándolos en **lotes (batches)**. El modelo se actualiza incrementalmente con cada batch en lugar de ver todos los datos a la vez.

### ¿Cómo funciona con cada boosting?

| Algoritmo | Mecanismo OOC | Parámetro clave |
|-----------|--------------|------------------|
| XGBoost | `xgb.train()` con `xgb_model=modelo_anterior` | Continúa agregando árboles |
| LightGBM | `lgb.train()` con `init_model=modelo_anterior` | Continúa desde el último árbol |
| CatBoost | `.fit()` con `init_model=modelo_anterior` | Idem |
| SGDRegressor | `.partial_fit()` | Actualiza pesos por gradiente estocástico |

La clave es que cada batch **agrega nuevos árboles** al ensemble existente, sin olvidar lo aprendido en batches anteriores.

In [ ]:
from src.data.ingestion import fetch_data_in_batches
from src.features.build_features import clean_dataframe, run_feature_engineering

print('=== OUT-OF-CORE TRAINING ===')
print(f'Modelo ganador: {WINNER_NAME}')
print('Iterando sobre analytics.train_set en batches de 500,000 filas...')
print('Cada batch: limpieza → feature engineering → actualizar modelo')
print()

# Query al train_set completo (800M filas)
query_train_full = "SELECT * FROM NYC_TAXI_P5.ANALYTICS.train_set"
BATCH_SIZE = 500_000  # ~200-300 MB por batch en RAM con 34 features

# El modelo OOC parte del ganador ya entrenado en la submuestra
# (warm start: ya tiene buenas hojas aprendidas, solo se refinan con más datos)
ooc_model = None
batch_num  = 0
total_rows = 0
ooc_rmse_history = []

for batch_df in fetch_data_in_batches(query_train_full, batch_size=BATCH_SIZE):
    batch_num += 1

    # ── Paso 1: Limpieza del batch ──
    batch_clean = clean_dataframe(batch_df)
    if len(batch_clean) == 0:
        print(f'  Batch {batch_num}: vacío después de limpieza, saltando.')
        continue

    # ── Paso 2: Feature Engineering del batch ──
    batch_fe = run_feature_engineering(batch_clean)

    # ── Paso 3: Preparar X e y ──
    batch_feats = [f for f in available_features if f in batch_fe.columns]
    X_batch = batch_fe[batch_feats]
    y_batch = batch_fe[TARGET]

    # ── Paso 4: Actualización incremental según el modelo ganador ──
    if WINNER_NAME in ('XGBoost',):
        X_proc = preprocessor_trees.transform(X_batch)
        dtrain = xgb.DMatrix(X_proc, label=y_batch)
        params = {
            'objective': 'reg:squarederror',
            'learning_rate': 0.05, 'max_depth': 6,
            'subsample': 0.8, 'colsample_bytree': 0.8,
            'reg_alpha': 0.1, 'reg_lambda': 1.0,
            'tree_method': 'hist', 'nthread': -1,
        }
        # xgb_model=ooc_model continúa desde el modelo anterior
        ooc_model = xgb.train(params, dtrain, num_boost_round=30,
                              xgb_model=ooc_model, verbose_eval=False)

    elif WINNER_NAME in ('LightGBM',):
        X_proc = preprocessor_trees.transform(X_batch)
        dtrain = lgb.Dataset(X_proc, label=y_batch)
        params = {
            'objective': 'regression', 'metric': 'rmse',
            'learning_rate': 0.05, 'num_leaves': 63,
            'min_child_samples': 50, 'subsample': 0.8,
            'colsample_bytree': 0.8, 'n_jobs': -1,
            'verbosity': -1,
        }
        # init_model continúa desde donde quedó
        ooc_model = lgb.train(params, dtrain, num_boost_round=30,
                              init_model=ooc_model,
                              callbacks=[lgb.log_evaluation(period=-1)])

    elif WINNER_NAME in ('CatBoost',):
        for col in cat_feature_names:
            if col in X_batch.columns:
                X_batch[col] = X_batch[col].astype(str).fillna('Unknown')
        pool = Pool(X_batch, label=y_batch, cat_features=cat_feature_names)
        if ooc_model is None:
            ooc_model = CatBoostRegressor(
                iterations=30, learning_rate=0.05, depth=6,
                l2_leaf_reg=3.0, eval_metric='RMSE',
                random_seed=RANDOM_STATE, verbose=0
            )
            ooc_model.fit(pool)
        else:
            ooc_model.fit(pool, init_model=ooc_model)

    else:
        # Fallback: SGDRegressor con partial_fit para cualquier otro modelo
        X_proc = preprocessor_trees.transform(X_batch)
        if ooc_model is None:
            ooc_model = SGDRegressor(loss='squared_error', learning_rate='invscaling',
                                     eta0=0.01, random_state=RANDOM_STATE)
        ooc_model.partial_fit(X_proc, y_batch)

    total_rows += len(X_batch)
    print(f'  ✅ Batch {batch_num:>3} | {len(X_batch):>8,} filas | Acumulado: {total_rows:>12,}')

print(f'\n🏁 Out-of-Core completado. Total filas procesadas: {total_rows:,}')

---
## 16. Exportar el Modelo Final

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

# Exportar preprocessor (necesario para la API)
joblib.dump(preprocessor_trees, '../models/preprocessor.pkl')
print('✅ Preprocessor guardado: models/preprocessor.pkl')

# Exportar modelo OOC final
if WINNER_NAME == 'XGBoost':
    ooc_model.save_model('../models/price_model.json')
    print('✅ Modelo XGBoost guardado: models/price_model.json')

elif WINNER_NAME == 'LightGBM':
    ooc_model.save_model('../models/price_model.txt')
    print('✅ Modelo LightGBM guardado: models/price_model.txt')

elif WINNER_NAME == 'CatBoost':
    ooc_model.save_model('../models/price_model.cbm')
    print('✅ Modelo CatBoost guardado: models/price_model.cbm')

else:
    joblib.dump(ooc_model, '../models/price_model.pkl')
    print('✅ Modelo guardado: models/price_model.pkl')

# Guardar también el modelo submuestra como .pkl (por compatibilidad con la API del profe)
joblib.dump(winner_model, '../models/price_model_sample.pkl')
print('✅ Modelo (submuestra) guardado: models/price_model_sample.pkl')

# Guardar configuración de features para la API
with open('../models/feature_config.json', 'w') as f:
    json.dump({
        **feat_cfg,
        'winner_model': WINNER_NAME,
        'cat_features': cat_feature_names,
        'available_features': available_features,
    }, f, indent=2)
print('✅ Configuración guardada: models/feature_config.json')

---
## 17. Resumen Final

Esta celda consolida los resultados para el reporte y la defensa.

In [ ]:
print('=' * 65)
print('RESUMEN FINAL DEL MODELADO')
print('=' * 65)

print(f"""
DATOS
  Train set:      ~800M filas (2015-2023, Snowflake)
  Submuestra EDA: {len(df):,} filas (1% del train_set)
  Val set:        2023 (dentro de la submuestra)
  Test set:       2025 (Snowflake, tocado una sola vez)

FEATURES
  Total features:  {len(available_features)}
  Temporales:      is_weekend, is_rush_hour, is_night, is_holiday,
                   pickup_hour_sin/cos, day_of_week_sin/cos, month_sin/cos
  Geográficas:     is_airport_trip, is_jfk, is_newark,
                   same_borough, is_inter_borough, is_manhattan_origin
  Interacciones:   rush_airport, night_airport, night_manhattan,
                   distance_per_passenger, is_long/short_trip

MODELOS EVALUADOS
  - Regresión Lineal (Baseline)
  - Voting Regressor (Ridge + DecisionTree + RandomForest)
  - Bagging (bootstrap=True)
  - Pasting (bootstrap=False)
  - AdaBoost
  - Gradient Boosting (GBDT)
  - XGBoost
  - LightGBM
  - CatBoost

RESULTADO
  Ganador (RMSE Val más bajo): {ganador['Modelo']}
  RMSE Val:   ${ganador['RMSE Val']:.4f}
  RMSE Test:  ${rmse_test:.4f}
  MAE Test:   ${mae_test:.4f}
  R² Test:    {r2_test:.4f}

ENTRENAMIENTO OUT-OF-CORE
  Mecanismo: batches de {BATCH_SIZE:,} filas desde Snowflake
  Filas procesadas: {total_rows:,}
  API de incremento: init_model / xgb_model (agrega árboles sin olvidar)
""")

print('Tabla comparativa final:')
print(results_df[['Modelo','RMSE Train','RMSE Val','MAE Val','R² Val']].to_string(index=False))